# Companion Notebook: Advanced Data Visualization

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/examples/14_advanced_data_viz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook follows the content from *Chapter 14: Advanced Data Visualization*. It covers three libraries — Seaborn for statistical visualization, Matplotlib for publication-quality figures, and Bokeh for interactive web-ready charts — applied to the Complete Journey retail dataset.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from completejourney_py import get_data

cj_data = get_data()
transactions = cj_data['transactions']
products     = cj_data['products']
demographics = cj_data['demographics']

df = (
    transactions
    .merge(products, on='product_id', how='left')
    .merge(demographics, on='household_id', how='left')
)

df.head()

---

## Seaborn

Seaborn is a statistical visualization library built on Matplotlib. It works natively with DataFrames and produces beautiful output with minimal code. Use it when you want to understand distributions, compare groups, or explore relationships between variables.

### Example 1: Visualizing Distributions

`sns.histplot` produces a histogram with an optional KDE overlay. Here we look at basket-level spending — one row per shopping trip.

In [ ]:
basket_spend = (
    transactions
    .groupby('basket_id', as_index=False)['sales_value']
    .sum()
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    basket_spend['sales_value'],
    bins=60,
    kde=True,
    ax=ax
)
ax.set_xlabel('Basket Spend ($)')
ax.set_title('Distribution of Basket-Level Spending')
plt.tight_layout()

### Example 2: Comparing Groups

`sns.boxplot` compares a numeric variable across categories. The `order` parameter controls how categories appear on the x-axis.

In [ ]:
household_spend = (
    df
    .groupby(['household_id', 'income'], as_index=False)['sales_value']
    .sum()
    .dropna(subset=['income'])
)

income_order = [
    'Under 15K', '15-24K', '25-34K', '35-49K',
    '50-74K', '75-99K', '100-124K', '125-149K',
    '150-174K', '175-199K', '200-249K', '250K+'
]

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(
    data=household_spend,
    x='income',
    y='sales_value',
    order=income_order,
    ax=ax
)
ax.set_xlabel('Income Range')
ax.set_ylabel('Total Household Spend ($)')
ax.set_title('Annual Household Spending by Income Range')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

### Example 3: Exploring Relationships

`sns.scatterplot` with `hue` maps a third variable to color, revealing whether a pattern differs across groups. Log scales spread the points when the data spans many orders of magnitude.

In [ ]:
store_summary = (
    df
    .groupby('store_id', as_index=False)
    .agg(
        sales_value=('sales_value', 'sum'),
        quantity=('quantity', 'sum'),
        n_trips=('basket_id', 'nunique')
    )
)
store_summary['traffic'] = (
    store_summary['n_trips']
    .gt(store_summary['n_trips'].median())
    .map({True: 'Above Median', False: 'Below Median'})
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=store_summary,
    x='quantity',
    y='sales_value',
    hue='traffic',
    palette={'Above Median': '#e63946', 'Below Median': '#457b9d'},
    s=80,
    ax=ax
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Total Quantity Sold (log scale)')
ax.set_ylabel('Total Sales (log scale)')
ax.set_title('Store-Level Sales vs. Quantity (by Traffic Tier)')
plt.tight_layout()

### Example 4: Heatmap

`sns.heatmap` takes a pivot table and encodes values as color. Here we build a day-of-week × hour-of-day grid showing when shoppers are most active.

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

heatmap_data = (
    transactions
    .assign(
        day_of_week=transactions['transaction_timestamp'].dt.day_name(),
        hour_of_day=transactions['transaction_timestamp'].dt.hour
    )
    .groupby(['day_of_week', 'hour_of_day'])['basket_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    heatmap_data,
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Number of Trips'}
)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
ax.set_title('Shopping Traffic: Day × Hour of Day', fontsize=14, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()

---

## Matplotlib

Matplotlib is the foundation of Python visualization — Pandas and Seaborn both build on it. Use Matplotlib when you need complete control: annotations, multi-panel layouts, tick formatting, or publication-quality styling. The standard pattern is `fig, ax = plt.subplots()`, then build the chart on `ax`.

### Example 1: Annotated Line Chart

`ax.annotate()` places arrows and text directly on the chart. `mtick.StrMethodFormatter` formats tick labels as currency.

In [ ]:
import matplotlib.ticker as mtick
from datetime import date as dt

daily_sales = (
    df
    .set_index('transaction_timestamp')['sales_value']
    .resample('D')
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    'transaction_timestamp', 'sales_value',
    data=daily_sales,
    color='steelblue',
    linewidth=1.5
)

ax.set_title('Total Daily Sales Across All Stores', size=14)
ax.set_ylabel('Total Sales')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.grid(linestyle='dashed', alpha=0.4)

ax.annotate(
    'Christmas Eve',
    xy=([dt(2017, 12, 20), 24000]),
    xytext=([dt(2017, 9, 1), 23500]),
    arrowprops={'color': 'crimson', 'width': 1.5},
    color='crimson',
    size=10
)
plt.tight_layout()

### Example 2: Multi-Panel Figures

`plt.subplots(1, 2)` creates two panels side by side. `axes[0]` and `axes[1]` give independent control over each panel. `fig.suptitle()` adds a shared title across all panels.

In [ ]:
dept_sales = (
    df
    .groupby('department', as_index=False)['sales_value']
    .sum()
    .nlargest(10, 'sales_value')
    .sort_values('sales_value')
)

basket_spend = (
    transactions
    .groupby('basket_id')['sales_value']
    .sum()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

axes[0].barh(dept_sales['department'], dept_sales['sales_value'], color='steelblue')
axes[0].set_title('Top 10 Departments by Revenue')
axes[0].set_xlabel('Total Sales')
axes[0].xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

axes[1].hist(basket_spend.clip(upper=100), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Basket Spend')
axes[1].set_xlabel('Basket Spend ($, clipped at $100)')
axes[1].set_ylabel('Number of Baskets')

fig.suptitle('Complete Journey — Sales Overview', fontsize=15, fontweight='bold');

### Example 3: Publication-Ready Styling

`plt.style.use()` applies a built-in style globally. Always reset to `'default'` afterward to avoid side effects on subsequent plots.

In [ ]:
plt.style.use('fivethirtyeight')

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

median_by_day = (
    df
    .assign(day_of_week=df['transaction_timestamp'].dt.day_name())
    .groupby('day_of_week')['sales_value']
    .median()
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(median_by_day.index, median_by_day.values, color='steelblue')
ax.set_title('Median Transaction Value by Day of Week')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.2f}'))
plt.xticks(rotation=0)
plt.tight_layout()

plt.style.use('default')

---

## Bokeh

Bokeh produces interactive charts that run in a web browser — users can zoom, pan, hover for details, and filter. The core pattern is: create a `figure`, add glyphs (visual marks), configure a `HoverTool`, then call `show()`.

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.io import output_notebook

output_notebook()

### Example 1: Interactive Line Chart with Hover

`ColumnDataSource` wraps a DataFrame and links it to the chart so the `HoverTool` can access column values by name. The `formatters` dict controls how datetime and numeric values are displayed in tooltips.

In [ ]:
daily_sales = (
    df
    .set_index('transaction_timestamp')['sales_value']
    .resample('D')
    .sum()
    .reset_index()
)

source = ColumnDataSource(daily_sales)

p = figure(
    title='Total Daily Sales (Hover to Explore)',
    x_axis_type='datetime',
    width=750, height=350,
    tools='pan,wheel_zoom,box_zoom,reset'
)

p.line(
    'transaction_timestamp', 'sales_value',
    source=source,
    line_width=2,
    color='steelblue',
    legend_label='Daily Sales'
)

hover = HoverTool(tooltips=[
    ('Date',  '@transaction_timestamp{%F}'),
    ('Sales', '@sales_value{$0,0.00}')
], formatters={'@transaction_timestamp': 'datetime'})
p.add_tools(hover)

p.legend.location = 'top_left'
p.xaxis.axis_label = 'Date'
p.yaxis.axis_label = 'Total Sales ($)'

show(p)

### Example 2: Interactive Scatter with Color Encoding

`factor_cmap` maps a categorical column to a color palette. Log axes spread the points when data spans many orders of magnitude. The hover tooltip reveals store-level details on demand.

In [ ]:
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10

store_summary = (
    df
    .groupby('store_id', as_index=False)
    .agg(
        sales_value=('sales_value', 'sum'),
        quantity=('quantity', 'sum'),
        n_trips=('basket_id', 'nunique')
    )
)
store_summary['traffic'] = (
    store_summary['n_trips']
    .gt(store_summary['n_trips'].median())
    .map({True: 'Above Median', False: 'Below Median'})
)
store_summary['store_id'] = store_summary['store_id'].astype(str)

source = ColumnDataSource(store_summary)
tiers = store_summary['traffic'].unique().tolist()

p = figure(
    title='Store-Level Sales vs. Quantity',
    x_axis_label='Total Quantity Sold (log scale)',
    y_axis_label='Total Sales (log scale)',
    x_axis_type='log',
    y_axis_type='log',
    width=650, height=420,
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p.scatter(
    x='quantity', y='sales_value',
    source=source,
    size=9,
    color=factor_cmap('traffic', palette=Category10[3][:2], factors=tiers),
    legend_field='traffic',
    alpha=0.7,
    line_color='white',
    line_width=0.5
)

hover = HoverTool(tooltips=[
    ('Store',    '@store_id'),
    ('Sales',    '@sales_value{$0,0}'),
    ('Quantity', '@quantity{0,0}'),
    ('Trips',    '@n_trips{0,0}')
])
p.add_tools(hover)

p.legend.location = 'top_left'
show(p)

### Example 3: Interactive Heatmap

`LinearColorMapper` maps a continuous variable to a color palette. `p.rect()` draws one rectangle per row in the DataFrame. Hovering over any cell shows the exact trip count for that day-hour combination.

In [ ]:
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.transform import transform
from bokeh.palettes import YlOrRd9

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hours = [str(h) for h in range(24)]

heatmap_df = (
    transactions
    .assign(
        day_of_week=transactions['transaction_timestamp'].dt.day_name(),
        hour_of_day=transactions['transaction_timestamp'].dt.hour.astype(str)
    )
    .groupby(['day_of_week', 'hour_of_day'])['basket_id']
    .nunique()
    .reset_index()
    .rename(columns={'basket_id': 'n_trips'})
)
heatmap_df['hour_label'] = heatmap_df['hour_of_day'].astype(int).apply(lambda h: f'{h}:00')

source = ColumnDataSource(heatmap_df)

mapper = LinearColorMapper(
    palette=YlOrRd9[::-1],
    low=heatmap_df['n_trips'].min(),
    high=heatmap_df['n_trips'].max()
)

p = figure(
    title='Shopping Traffic: Day × Hour of Day',
    x_range=hours,
    y_range=day_order[::-1],
    x_axis_label='Hour of Day',
    width=800, height=320,
    tools='hover,save,reset',
    tooltips=[
        ('Day',   '@day_of_week'),
        ('Hour',  '@hour_label'),
        ('Trips', '@n_trips{0,0}')
    ]
)

p.rect(
    x='hour_of_day', y='day_of_week',
    width=1, height=1,
    source=source,
    fill_color=transform('n_trips', mapper),
    line_color=None
)

color_bar = ColorBar(color_mapper=mapper, label_standoff=8, title='Trips')
p.add_layout(color_bar, 'right')

p.xaxis.major_label_overrides = {str(h): f'{h}:00' for h in range(0, 24, 3)}

show(p)